## Download Images

In [ ]:
import ee
import geemap
import os
import glob
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import cv2
from skimage.filters import threshold_otsu, sobel
from skimage import measure
import matplotlib.cm as cm

# ==========================================
# TASK 1 & 2: Initialize & Download SAR Data
# ==========================================

# 1. Initialize Earth Engine with your Project ID
try:
    ee.Initialize(project='grand-century-284413')
except Exception as e:
    ee.Authenticate() # Will prompt you to log in if not already authenticated
    ee.Initialize(project='grand-century-284413')

# Define the Region of Interest (ROI) - Wollongong Beach
coords = [
    [150.90936984042523, -34.39635225871997],
    [150.93923892001507, -34.39635225871997],
    [150.93923892001507, -34.33542219483208],
    [150.90936984042523, -34.33542219483208],
    [150.90936984042523, -34.39635225871997]
]
roi = ee.Geometry.Polygon(coords)

# Fetch Sentinel-1 SAR GRD images
# Filtering for VH polarization, Interferometric Wide (IW) swath mode, and descending orbits
collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(roi) \
    .filterDate('2018-01-01', '2018-12-31') \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .select('VH')

# To avoid downloading hundreds of overlapping tiles, let's create monthly composites (medians)
# This reduces speckle noise and gives us 12 clean images for the time series.
months = ee.List.sequence(1, 12)
def create_monthly_composite(month):
    start = ee.Date.fromYMD(2018, month, 1)
    end = start.advance(1, 'month')
    composite = collection.filterDate(start, end).median().set('month', month)
    return composite.clip(roi)

monthly_collection = ee.ImageCollection(months.map(create_monthly_composite))

# Download the images to a local directory
out_dir = os.path.join(os.getcwd(), 'SAR_Coastline_Data')
os.makedirs(out_dir, exist_ok=True)

print("Downloading monthly SAR composites. This may take a moment...")
geemap.ee_export_image_collection(
    monthly_collection, 
    out_dir=out_dir, 
    scale=10, 
    region=roi, 
    file_per_band=False
)
print("Download complete!")

## Segmentation Algorithms

In [ ]:
# ==========================================
# TASK 3: Dynamic Edge-Otsu Algorithm
# ==========================================

def dynamic_edge_otsu(image_array):
    """
    Applies an Edge-Otsu threshold. Standard Otsu fails if water/land ratio is highly unbalanced.
    This finds high-gradient edges (the coastline) and calculates the Otsu threshold 
    ONLY using the pixels located around that boundary.
    """
    # 1. Apply Gaussian Blur to reduce SAR speckle noise
    blurred = cv2.GaussianBlur(image_array, (5, 5), 0)
    
    # 2. Compute edges using Sobel filter
    edges = sobel(blurred)
    
    # 3. Create a mask of the top 10% strongest edges (likely the land-water boundary)
    edge_thresh = np.percentile(edges, 90)
    edge_mask = edges > edge_thresh
    
    # 4. Extract pixel values along the boundary
    boundary_pixels = blurred[edge_mask]
    
    # 5. Calculate Otsu threshold on the boundary pixels
    if len(boundary_pixels) > 0:
        otsu_val = threshold_otsu(boundary_pixels)
    else:
        otsu_val = threshold_otsu(blurred) # Fallback
        
    # 6. Segment image (Water is darker/lower backscatter than land)
    binary_segmented = blurred < otsu_val
    return binary_segmented, otsu_val, blurred


In [ ]:
import numpy as np
import cv2
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from skimage.filters import threshold_otsu

# ==========================================
# TASK 3: Peak and Valley Algorithm
# ==========================================

def peak_valley_threshold(image_array):
    """
    Segments an image by finding the two largest peaks in its smoothed histogram
    and setting the threshold at the deepest valley between them.
    """
    # 1. Apply Gaussian Blur to reduce SAR speckle noise
    blurred = cv2.GaussianBlur(image_array, (5, 5), 0)
    
    # Extract valid pixels (ignoring NaNs that might come from GEE edges)
    valid_pixels = blurred[~np.isnan(blurred)]
    
    # 2. Compute the histogram
    min_val, max_val = np.min(valid_pixels), np.max(valid_pixels)
    hist, bin_edges = np.histogram(valid_pixels, bins=256, range=(min_val, max_val))
    
    # 3. Smooth the 1D histogram to remove minor local fluctuations (noise)
    # This is critical for SAR; otherwise, find_peaks finds dozens of micro-peaks
    smoothed_hist = gaussian_filter1d(hist, sigma=5)
    
    # 4. Find the peaks in the smoothed histogram
    # Prominence helps ensure we only grab significant peaks, not minor bumps
    peaks, _ = find_peaks(smoothed_hist, prominence=100) 
    
    # Fallback safety: If the image isn't clearly bimodal, default back to standard Otsu
    if len(peaks) < 2:
        print("  -> Warning: Could not find two distinct peaks. Falling back to Otsu.")
        otsu_val = threshold_otsu(valid_pixels)
        return blurred < otsu_val, otsu_val, blurred
        
    # 5. Isolate the two largest peaks
    # Sort peaks by their frequency count (height) in descending order, grab the top 2
    largest_peaks = sorted(peaks, key=lambda x: smoothed_hist[x], reverse=True)[:2]
    
    # Sort the two peaks spatially (x-axis) so peak1 is water (dark) and peak2 is land (bright)
    peak1, peak2 = sorted(largest_peaks)
    
    # 6. Find the valley (minimum) strictly between these two peaks
    valley_idx_relative = np.argmin(smoothed_hist[peak1:peak2])
    valley_idx_absolute = peak1 + valley_idx_relative
    
    # Convert the bin index back to an actual image intensity value
    threshold_val = bin_edges[valley_idx_absolute]
    
    # 7. Segment the image (Water is darker/lower backscatter than the threshold)
    binary_segmented = blurred < threshold_val
    
    return binary_segmented, threshold_val, blurred

## Processing and Exporting Images

In [ ]:
import os
import glob
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from skimage import measure

# ==========================================
# TASK 4 & 5: Processing, Plotting & Exporting Segmented Images
# ==========================================

tif_files = sorted(glob.glob(os.path.join(out_dir, '*.tif')))
all_shorelines = [] # To store contours for Task 6

# Create a new directory specifically for the segmented image exports
seg_export_dir = os.path.join(os.getcwd(), 'SAR_Segmented_Masks')
os.makedirs(seg_export_dir, exist_ok=True)

# Loop through all available TIFF files
for idx, tif in enumerate(tif_files):
    print(f"Processing Image {idx + 1} of {len(tif_files)}...")
    
    with rasterio.open(tif) as src:
        raw_img = src.read(1)
        
        # Copy the spatial metadata so we can use it to save our segmented image later
        meta = src.meta.copy() 
        
        # Filter out nodata/NaN values or infinite values common in GEE exports
        raw_img = np.nan_to_num(raw_img, nan=np.nanmean(raw_img))

    # Perform segmentation (Ensure you use whichever function you settled on, e.g., peak_valley_threshold)
    segmented_img, threshold_val, smoothed_img = peak_valley_threshold(raw_img)
    # segmented_img, threshold_val, smoothed_img = kmeans_segmentation(raw_img, k=3)

    # ---------------------------------------------------------
    # NEW: Exporting the Segmented Image
    # ---------------------------------------------------------
    # The segmented image is usually a boolean array (True/False). 
    # We convert it to uint8 and multiply by 255 so it exports as standard Black (0) and White (255) pixels.
    seg_img_export = (segmented_img.astype('uint8') * 255)
    
    # Update the metadata to match our new uint8 2D array
    meta.update({
        "dtype": 'uint8',
        "nodata": None  # Optional: remove nodata flag for a pure black/white mask
    })
    
    # Define the output path for this specific month
    out_tif_path = os.path.join(seg_export_dir, f'Segmented_Month_{idx + 1:02d}.tif')
    
    # Write the segmented array to a new GeoTIFF
    with rasterio.open(out_tif_path, 'w', **meta) as dst:
        dst.write(seg_img_export, 1)
        
    print(f"  -> Exported segmented mask to: {out_tif_path}")
    # ---------------------------------------------------------

    # Extract Shoreline (Line Polygon)
    contours = measure.find_contours(segmented_img, 0.5)
    
    # Safety check: skip plotting if no contours were found in a noisy image
    if not contours:
        print(f"  -> Warning: No coastline detected in Month {idx + 1}. Skipping plot.")
        continue
        
    # Assume the longest contour is our continuous coastline
    main_shoreline = max(contours, key=len)
    all_shorelines.append(main_shoreline)

    # Plotting (Tasks 2, 4, and 5 combined)
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # 1. Raw VH Image
    axes[0].imshow(raw_img, cmap='gray')
    axes[0].set_title(f'Raw VH SAR Image (Month {idx + 1})')
    axes[0].axis('off')

    # 2. Segmented Image with Shoreline Overlay
    axes[1].imshow(segmented_img, cmap='gray')
    axes[1].plot(main_shoreline[:, 1], main_shoreline[:, 0], color='red', linewidth=2)
    axes[1].set_title(f'Segmented Image & Shoreline')
    axes[1].axis('off')

    # 3. Isolated Shoreline Polyline
    axes[2].plot(main_shoreline[:, 1], main_shoreline[:, 0], color='blue', linewidth=2)
    axes[2].set_xlim(0, raw_img.shape[1])
    axes[2].set_ylim(raw_img.shape[0], 0) # Invert Y to match image coordinates
    axes[2].set_title(f'Extracted Polyline')
    axes[2].set_aspect('equal')
    axes[2].axis('off')

    # 4. Histogram
    axes[3].hist(smoothed_img.ravel(), bins=100, color='gray', alpha=0.7)
    axes[3].axvline(threshold_val, color='red', linestyle='dashed', linewidth=2, label=f'Threshold: {threshold_val:.2f}')
    axes[3].set_title(f'Histogram (Month {idx + 1})')
    axes[3].legend()

    # Add a main title for the whole row
    plt.suptitle(f'Coastline Analysis: Wollongong Beach - Month {idx + 1}', fontsize=16)
    plt.tight_layout()
    plt.show()

## Time Series Mapping

In [ ]:
# ==========================================
# TASK 6: Time Series Mapping
# ==========================================

plt.figure(figsize=(10, 10))
plt.title('Time Series Coastline Mapping (2018)')

# Generate a color map for the 12 months
colors = cm.jet(np.linspace(0, 1, len(tif_files)))

for idx, tif in enumerate(tif_files):
    with rasterio.open(tif) as src:
        img = src.read(1)
        img = np.nan_to_num(img, nan=np.nanmean(img))
        
        # Segment and extract
        seg, _, _ = dynamic_edge_otsu(img)
        conts = measure.find_contours(seg, 0.5)
        
        if conts:
            longest = max(conts, key=len)
            plt.plot(longest[:, 1], longest[:, 0], color=colors[idx], label=f'Month {idx+1}', linewidth=1.5)

plt.xlim(0, raw_img.shape[1])
plt.ylim(raw_img.shape[0], 0)
plt.legend(loc='upper right', bbox_to_anchor=(1.25, 1))
plt.axis('off')
plt.show()